# 인코딩과 데이터 분리

독창적 가상 데이터와 자체 연습 문항입니다. 제공 샘플의 흐름을 참고했으며 원문 복제가 아닙니다.

14문항 · 파트 연습 · 힌트 사용 가능

CSV 두 개와 노트북을 같은 폴더에 두세요. 커널 재시작 → 전체 실행 → 저장한 뒤 사이트에 제출하세요. 각 답안 다음 검증 셀은 실행 결과를 검사합니다. 검증 셀을 바꾸면 점수 신뢰도가 사라집니다. 공식 점수 예측이 아닌 자체 연습 점수입니다.

데이터: ticket_id 식별번호, created_at 접수일시, issue_type 문의유형, contact_channel 채널, membership_tier 등급, tenure_months 가입개월, monthly_spend 월금액(원), previous_tickets 과거 문의 수, wait_minutes 대기분, past_satisfaction_score 과거만족도(1~5), escalated 이관여부(0/1), resolution_hours 해결시간(시간). resolution_hours는 분류 입력에서 제외합니다.

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, mean_absolute_error
SET_ID='part-encoding'
DATA_FILE="practice.csv"
MEMBERS_FILE="members.csv"
MODEL_FILE="best_part-encoding.keras"

from pathlib import Path
import json as _json
from IPython.display import display as _display
_raw = pd.read_csv(DATA_FILE)
_grade_results = []
def _filled(d):
    x=d.copy()
    cols=['wait_minutes','past_satisfaction_score','monthly_spend']
    x[cols]=x[cols].fillna(x[cols].median())
    x['contact_channel']=x.contact_channel.fillna(x.contact_channel.mode().iloc[0])
    return x
def _plot_ok(fig,h,k1,k2,title,ylabel,labels=None):
    a=fig.axes[0]
    ls=a.get_lines()
    labels=labels or [k1,k2]
    return (len(ls)==2 and a.get_title()==title and a.get_xlabel()=='Epoch' and a.get_ylabel()==ylabel
      and np.allclose(ls[0].get_ydata(),h.history[k1]) and np.allclose(ls[1].get_ydata(),h.history[k2])
      and a.get_legend() is not None and [t.get_text() for t in a.get_legend().get_texts()]==labels)
def _grade_q(number,checks):
    items=[]
    for label,expr in checks:
        try:
            ok=bool(eval(expr,globals()))
            items.append({'label':label,'passed':ok,'detail':'검사 통과' if ok else '실행 결과가 지시와 다릅니다. 변수명·조건·파라미터를 확인하세요.'})
        except Exception as e:
            items.append({'label':label,'passed':False,'detail':type(e).__name__+': '+str(e)[:180]})
    report={'schema':'aice-lab-v1','setId':SET_ID,'question':number,'checks':items}
    _grade_results.append(report)
    _display({'application/json':report},raw=True)
data=pd.read_csv(DATA_FILE)
clean = data.drop_duplicates().copy()
clean = clean.loc[clean['wait_minutes'].isna() | (clean['wait_minutes'] < 300)].reset_index(drop=True)
num = ['wait_minutes','past_satisfaction_score','monthly_spend']
clean[num] = clean[num].fillna(clean[num].median())
clean['contact_channel'] = clean['contact_channel'].fillna(clean['contact_channel'].mode().iloc[0])


## 1. 라벨 인코딩 (7점)

LabelEncoder를 le에 생성하고 clean의 membership_tier를 변환하여 tier_encoded에 저장하세요. clean은 수정하지 마세요.

In [40]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

tier_encoded = le.fit_transform(clean["membership_tier"])

tier_encoded

array([0, 1, 1, 0, 0, 2, 2, 0, 2, 1, 1, 1, 0, 1, 0, 0, 1, 2, 2, 2, 2, 0,
       2, 2, 2, 0, 1, 1, 2, 2, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 2, 0,
       0, 2, 1, 1, 0, 2, 2, 1, 0, 0, 0, 1, 2, 0, 2, 2, 2, 0, 1, 0, 1, 0,
       0, 2, 2, 1, 2, 2, 2, 2, 0, 2, 0, 0, 2, 0, 0, 2, 0, 0, 1, 2, 0, 1,
       1, 2, 1, 1, 0, 2, 0, 0, 2, 0, 2, 1, 1, 0, 2, 2, 2, 2, 1, 1, 1, 1,
       1, 0, 0, 0, 2, 0, 1, 0, 1, 0, 0, 0, 2, 1, 0, 2, 2, 0, 0, 2, 2, 1,
       0, 1, 2, 0, 2, 2, 0, 0, 2, 2, 1, 1, 1, 2, 1, 2, 0, 1, 1, 0, 2, 1,
       2, 2, 2, 1, 2, 2, 0, 1, 2, 0, 2, 1, 1, 2, 2, 1, 2, 2, 2, 0, 0, 1,
       2, 1, 2, 0, 1, 0, 1, 2, 1, 1, 2, 1, 1, 0, 1, 0, 2, 2, 2, 0, 0, 2,
       2, 1, 1, 2, 1, 0, 1, 2, 1, 1, 0, 2, 1, 0, 2, 2, 0, 1, 1, 0, 1, 0,
       2, 1, 0, 1, 0, 0, 1, 0, 0, 2, 2, 1, 2, 2, 0, 2, 2, 2, 1, 1, 1, 0,
       0, 0, 2, 0, 0, 1, 1, 1, 1, 1, 2, 1, 1, 2, 0, 2, 2, 2, 2, 1, 1, 1,
       1, 1, 2, 2, 2, 1, 0, 1, 1, 0, 0, 1, 0, 2, 2, 1, 1, 1, 2, 1, 1, 2,
       2, 2, 0, 2, 2, 1, 1, 2, 2, 0, 0, 2, 1, 1, 1,

In [41]:
_grade_q(1, [['인코딩 배열', 'np.array_equal(tier_encoded,LabelEncoder().fit_transform(clean.membership_tier))'], ['클래스 정렬', 'np.array_equal(le.classes_,["Basic","Gold","Silver"])']])

## 2. 대응표 확인 (7점)

le의 클래스 배열을 tier_classes에 복사하세요.

In [42]:
# 여기에 답안코드를 작성하고 실행하세요.
tier_classes = le.classes_
tier_classes

array(['Basic', 'Gold', 'Silver'], dtype=object)

In [43]:
_grade_q(2, [['클래스 배열', 'np.array_equal(tier_classes,["Basic","Gold","Silver"])']])

## 3. 역변환 (7점)

le로 숫자 [0,2]를 원래 값으로 복원하여 restored에 저장하세요.

In [44]:
# 여기에 답안코드를 작성하고 실행하세요.
restored = le.inverse_transform([0,2])
restored

array(['Basic', 'Silver'], dtype=object)

In [45]:
_grade_q(3, [['역변환 결과', 'list(restored)==["Basic","Silver"]']])

## 4. 같은 인코더 재학습 (7점)

le로 clean.contact_channel을 fit_transform하고 channel_encoded에 저장하세요. 마지막 대응표를 last_classes에 복사하세요.

In [46]:
# 여기에 답안코드를 작성하고 실행하세요.
channel_encoded = le.fit_transform(clean["contact_channel"])
last_classes = le.classes_

In [47]:
_grade_q(4, [['채널 인코딩', 'np.array_equal(channel_encoded,LabelEncoder().fit_transform(clean.contact_channel))'], ['마지막 대응표', 'list(last_classes)==["Chat","Email","Phone"]']])

## 5. 반복문 상태 추적 (7점)

clean의 복사본 loop_data를 만들고 channel → membership 순서로 같은 loop_le로 인코딩하세요. 마지막 classes_를 loop_classes에 복사하세요.

In [48]:
# 여기에 답안코드를 작성하고 실행하세요.
loop_data = clean.copy()

loop_cols = ["contact_channel", "membership_tier"]
loop_le = LabelEncoder()

for i in loop_cols:
    loop_data[i] = loop_le.fit_transform(loop_data[i])
loop_classes = loop_le.classes_  

In [49]:
_grade_q(5, [['반복문 종료 상태', 'list(loop_classes)==["Basic","Gold","Silver"]'], ['두 컬럼 변환', 'np.array_equal(loop_data.contact_channel,channel_encoded) and np.array_equal(loop_data.membership_tier,tier_encoded)']])

## 6. 컬럼별 인코더 보관 (7점)

encoders 딕셔너리에 contact_channel, membership_tier별 별도 인코더를 학습하여 보관하세요.

In [50]:
# 여기에 답안코드를 작성하고 실행하세요.
encoders = {}
encode_data = clean.copy()

for i in loop_cols:
    encode_le = LabelEncoder()
    encode_data[i] = encode_le.fit_transform(encode_data[i])
    encoders[i] = encode_le

encoders

{'contact_channel': LabelEncoder(), 'membership_tier': LabelEncoder()}

In [51]:
_grade_q(6, [['서로 다른 객체', 'encoders["contact_channel"] is not encoders["membership_tier"]'], ['대응표 유지', 'list(encoders["contact_channel"].classes_)==["Chat","Email","Phone"] and list(encoders["membership_tier"].classes_)==["Basic","Gold","Silver"]']])

## 7. transform만 적용 (7점)

저장한 membership 인코더로 ["Silver","Basic"]을 transform하여 new_tiers에 저장하세요.

In [52]:
# 여기에 답안코드를 작성하고 실행하세요.
new_tiers = encoders["membership_tier"].transform(["Silver", "Basic"])
new_tiers

array([2, 0])

In [53]:
_grade_q(7, [['기존 기준 변환', 'list(new_tiers)==[2,0]']])

## 8. 이진 map (7점)

clean.escalated를 0→No, 1→Yes로 변환한 label_words를 만드세요.

In [54]:
# 여기에 답안코드를 작성하고 실행하세요.
label_words = clean["escalated"].map({
    0:"No",
    1:"Yes"
})

In [55]:
_grade_q(8, [['매핑 결과', 'label_words.equals(clean.escalated.map({0:"No",1:"Yes"}))']])

## 9. 원-핫 인코딩 (7점)

clean의 issue_type, contact_channel, membership_tier를 원-핫 인코딩하여 encoded에 저장하세요. 첫 범주를 유지하고 정수형을 사용하세요.

In [56]:
# 여기에 답안코드를 작성하고 실행하세요.
encoded = pd.get_dummies(
    clean,
    columns=["issue_type","contact_channel","membership_tier"],
    dtype=int
)

In [57]:
_grade_q(9, [['원핫 결과', 'encoded.equals(pd.get_dummies(clean,columns=["issue_type","contact_channel","membership_tier"],dtype=int))']])

## 10. X/y 분리 (7점)

encoded에서 ticket_id, created_at, resolution_hours, escalated를 제외해 X에 저장하고 escalated를 y에 저장하세요.

In [58]:
# 여기에 답안코드를 작성하고 실행하세요.
X = encoded.drop(columns=["ticket_id","created_at","resolution_hours","escalated"])
y = encoded["escalated"]

In [59]:
_grade_q(10, [['입력 컬럼', 'list(X.columns)==list(encoded.drop(columns=["ticket_id","created_at","resolution_hours","escalated"]).columns)'], ['타깃', 'y.equals(clean.escalated)']])

## 11. 계층 분리 (7점)

X,y를 70:30, random_state=7, stratify=y로 X_train,X_valid,y_train,y_valid에 분리하세요.

In [60]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=7,
    stratify=y
)

In [61]:
_grade_q(11, [['정확한 분할', 'np.array_equal(X_train.index,train_test_split(X,y,test_size=.3,random_state=7,stratify=y)[0].index) and np.array_equal(y_valid,train_test_split(X,y,test_size=.3,random_state=7,stratify=y)[3])']])

## 12. 표준화 (7점)

ss에 StandardScaler를 만들고 훈련은 fit_transform, 검증은 transform하여 X_train_scaled,X_valid_scaled에 저장하세요.

In [62]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()

X_train_scaled = ss.fit_transform(X_train)
X_valid_scaled = ss.transform(X_valid)

In [63]:
_grade_q(12, [['훈련 변환', 'np.allclose(X_train_scaled,StandardScaler().fit_transform(X_train))'], ['검증 같은 기준', 'np.allclose(X_valid_scaled,StandardScaler().fit(X_train).transform(X_valid))']])

## 13. RobustScaler (7점)

rs에 RobustScaler를 만들고 훈련은 fit_transform해 train_robust, 검증은 transform해 valid_robust에 저장하세요.

In [64]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.preprocessing import RobustScaler

rs = RobustScaler()

train_robust = rs.fit_transform(X_train)
valid_robust = rs.transform(X_valid)

In [65]:
_grade_q(13, [['Robust 훈련', 'np.allclose(train_robust,RobustScaler().fit_transform(X_train))'], ['Robust 검증', 'np.allclose(valid_robust,RobustScaler().fit(X_train).transform(X_valid))']])

## 14. 반올림 답안 (9점)

valid_robust의 전체 최댓값을 반올림하여 answer14에 저장하세요.

In [66]:
# 여기에 답안코드를 작성하고 실행하세요.
answer14 = round(np.max(valid_robust))

In [67]:
_grade_q(14, [['최댓값 반올림', 'answer14==round(np.max(RobustScaler().fit(X_train).transform(X_valid)))']])

## 제출
검증 결과를 포함한 상태로 저장하세요. VS Code에서 실행이 중단되면 오류를 고친 뒤 다시 전체 실행하세요. 설치가 필요하면 사이트의 환경 설정 안내를 확인하세요.